# Day 60：毕业设计开发 Day 2：企业级混合检索 RAG + Rerank + 引用溯源（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 9 周**：星辰智脑毕业设计全栈开发与上线  
> **GitLab 工单**：`STARMIND-060` ｜ **开发分支**：`feat/day60-capstone-hybrid-rag`  
> **配套 HTML5 课件**：`courseware/day60/index.html` ｜ **独立版本快照**：`versions/day60/`

---

## 🎙️ 老陈开场旁白

早啊各位！毕业设计 Day 2，干翻 RAG！面试官问 RAG 优化，你要是只答'切块塞向量库做 Top-K'，基本一面就挂了。为什么？因为纯向量检索对企业里的专有编号（比如报销单号 `FIN-2026`、错误码 `ERR_4291`、规章条款号）极度不敏感，而纯 BM25 又听不懂同义表达（比如'差旅住宿标准'和'出差住酒店上限'）。今天咱用纯 Python 把工业界最稳的四段式 RAG 流水线直接打通：中文 N-gram BM25 词法召回 + 轻量语义向量召回 -> RRF 倒数秩融合 -> 交叉特征 Rerank 精排 -> 强制 `[DOC-xx]` 脚标引用溯源与低分拒答防幻觉！

### 承上启下
- **昨日基线**：昨天我们搭好了星辰智脑的配置中心、JSON 日志与 SQLAlchemy 持久化底座，今天正式进入核心智能引擎第一战——企业级 RAG 知识库主链路。
- **明日预告**：明天（Day 61）我们开发核心多 Agent 编排主链路：把今天的混合检索 RAG 封装为专家节点，接入 LangGraph 状态机、MCP 工具、多轮记忆与人工审批（HITL）。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day60'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-060）

星辰智脑知识库承载公司财务报销、IT 运维、法务合同与产品 SLA 文档。业务要求回答必须有据可查、精准召回专有名词与同义问法，且在知识库无相关内容时明确拒答而非编造。

**今日迭代目标：**
- 实现双路混合检索器：BM25 词法打分 + 稠密词袋/语义余弦相似度。
- 实现 RRF（Reciprocal Rank Fusion）融合与 Rerank 精排重打分。
- 实现带强制 `[doc_id]` 引用角标及置信度阈值拒答机制的 RAG 问答组装器。

**验收标准：**
- 混合检索在专有编号查询和同义语义查询上均能将目标文档排入 Top-2。
- 当检索最高精排得分低于阈值时，明确返回'知识库未检索到依据'拒答提示，绝不瞎编。
- 纯 Python 离线运行，pytest 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 为什么 RRF 比加权线性相加更稳？

BM25 分数无上界（可能 0.5 也可能 25），而余弦相似度在 `[-1, 1]` 之间，直接 `0.5*bm25 + 0.5*vec` 会导致 BM25 完全淹没向量分。RRF 只看各路排序位次 `1/(k + rank)`（通常 `k=60`），天然免归一化，工程极稳。

> **⚠️ 老陈避坑**：坑点：把 RRF 后的微小数值（如 0.03）直接当成置信度阈值拿去做拒答判断，正确做法是用精排 Rerank 的 `[0, 1]` 归一化分做拒答门禁。

### 2. 2. 企业知识库切块的黄金经验

Markdown/Word 规章制度优先按标题层级做语义切块（保留父标题上下文，如'财务制度 > 差旅报销 > 住宿标准'），块长控制在 300~500 汉字，重叠（overlap）10%~15%，切忌硬按固定字符数把一个表格或条款拦腰截断。

> **⚠️ 老陈避坑**：坑点：切块丢失文档标题，导致检索出'不超过 600 元/晚'却不知道是一线城市还是二线城市的住宿标准。

### 3. 3. 强制引用溯源如何消除业务方信任危机？

在最终答案中强制标注 `[DOC-01]` 并在返回 JSON 中附带原文 `snippet` 和 `title`，前端渲染为可悬浮预览的原文卡片，业务方一眼核实来源，采纳率立涨 3 倍。

> **⚠️ 老陈避坑**：坑点：允许模型在没有检索到依据时自由发挥，导致财务制度问答出现'编造报销额度'的严重事故。


## 💻 实操步骤 1：步骤一：实现中文混合检索（BM25 + 语义向量 + RRF 融合）引擎

- **目标文件**：`versions/day60/src/day60_hybrid_retriever.py`
- **老陈代码旁白**：先手写纯 Python 零依赖的混合检索引擎：BM25 专打专有编号和精准词，语义向量通过领域概念映射覆盖同义问法，最后用 RRF 无量纲融合。


In [ ]:
"""
Day 60 - 模块一：星辰智脑混合检索引擎（BM25 + 语义向量 + RRF 融合）
"""
from __future__ import annotations

import math
import re
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Tuple

SYNONYM_GROUPS = [
    ("酒店", "住宿", "差旅", "宾馆"),
    ("报销", "发票", "费用", "核销"),
    ("报错", "异常", "故障", "失败", "错误码"),
    ("熔断", "限流", "超额", "429"),
    ("密码", "登录", "账号", "sso"),
]


def tokenize(text: str) -> List[str]:
    """中英文混合分词：英文/数字完整保留，中文采用 2-gram + 单字"""
    text_low = text.lower()
    alnum = re.findall(r"[a-z0-9_\-]+", text_low)
    cjk_runs = re.findall(r"[\u4e00-\u9fff]+", text_low)
    tokens: List[str] = list(alnum)
    for run in cjk_runs:
        tokens.extend(list(run))
        tokens.extend(run[i : i + 2] for i in range(len(run) - 1))
    return tokens


def semantic_expand(tokens: List[str]) -> Counter[str]:
    """轻量领域同义概念向量投影（生产可无缝替换为 bge-m3 Embedding）"""
    vec = Counter(tokens)
    joined = "".join(tokens)
    for idx, group in enumerate(SYNONYM_GROUPS):
        if any(word in joined for word in group):
            vec[f"__concept_{idx}__"] += 3.0
    return vec


def cosine_sim(v1: Counter[str], v2: Counter[str]) -> float:
    common = set(v1.keys()) & set(v2.keys())
    if not common:
        return 0.0
    dot = sum(v1[k] * v2[k] for k in common)
    n1 = math.sqrt(sum(v * v for v in v1.values()))
    n2 = math.sqrt(sum(v * v for v in v2.values()))
    return dot / max(1e-9, n1 * n2)


@dataclass
class KnowledgeChunk:
    doc_id: str
    title: str
    content: str


class HybridRetriever:
    """BM25 + 语义向量 + RRF 融合检索器"""

    def __init__(self, chunks: List[KnowledgeChunk]) -> None:
        self.chunks = chunks
        self._doc_tokens = [tokenize(f"{c.title} {c.content}") for c in chunks]
        self._doc_vecs = [semantic_expand(toks) for toks in self._doc_tokens]
        self._avg_dl = sum(len(t) for t in self._doc_tokens) / max(1, len(chunks))
        self._df: Counter[str] = Counter()
        for toks in self._doc_tokens:
            for term in set(toks):
                self._df[term] += 1

    def _bm25_scores(self, q_tokens: List[str], k1: float = 1.5, b: float = 0.75) -> List[Tuple[int, float]]:
        n_docs = len(self.chunks)
        scores = []
        for idx, d_toks in enumerate(self._doc_tokens):
            tf = Counter(d_toks)
            dl = len(d_toks)
            s = 0.0
            for qt in q_tokens:
                if qt not in tf:
                    continue
                df = self._df[qt]
                idf = math.log((n_docs - df + 0.5) / (df + 0.5) + 1.0)
                num = tf[qt] * (k1 + 1.0)
                den = tf[qt] + k1 * (1.0 - b + b * dl / max(1.0, self._avg_dl))
                s += idf * (num / den)
            scores.append((idx, s))
        return sorted(scores, key=lambda x: x[1], reverse=True)

    def _vector_scores(self, q_tokens: List[str]) -> List[Tuple[int, float]]:
        q_vec = semantic_expand(q_tokens)
        scores = [(idx, cosine_sim(q_vec, d_vec)) for idx, d_vec in enumerate(self._doc_vecs)]
        return sorted(scores, key=lambda x: x[1], reverse=True)

    def search_rrf(self, query: str, top_k: int = 3, rrf_k: int = 60) -> List[Tuple[KnowledgeChunk, float]]:
        q_tokens = tokenize(query)
        bm25_rank = self._bm25_scores(q_tokens)
        vec_rank = self._vector_scores(q_tokens)

        rrf_scores: Dict[int, float] = {}
        for rank, (idx, score) in enumerate(bm25_rank, start=1):
            if score > 0:
                rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (rrf_k + rank)
        for rank, (idx, score) in enumerate(vec_rank, start=1):
            if score > 0:
                rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (rrf_k + rank)

        ordered = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        return [(self.chunks[idx], round(sc, 5)) for idx, sc in ordered]


if __name__ == "__main__":
    kb = [
        KnowledgeChunk("DOC-01", "差旅住宿报销标准", "一线城市出差住宿上限650元/晚，需在费用中心上传电子专票核销。"),
        KnowledgeChunk("DOC-02", "网关错误码ERR_4291手册", "错误码ERR_4291表示部门日Token配额已触发100%硬熔断，请联系管理员提额。"),
    ]
    ret = HybridRetriever(kb)
    print("专有码检索:", [(c.doc_id, s) for c, s in ret.search_rrf("接口报 ERR_4291 怎么办")])
    print("同义词检索:", [(c.doc_id, s) for c, s in ret.search_rrf("出差住宾馆最多能花多少钱")])


### 🔍 步骤 1 代码深度解析

`tokenize` 同时保留完整英文/数字错误码和中文 2-gram，让 BM25 精准命中 `ERR_4291`；`semantic_expand` 在无外部 Embedding API 时也能通过概念投影把'住宾馆'映射到'差旅住宿'。

**预期运行输出参考：**
```text
专有码检索: [('DOC-02', 0.03279), ('DOC-01', 0.01613)]
同义词检索: [('DOC-01', 0.03279), ('DOC-02', 0.01613)]
```


## 💻 实操步骤 2：步骤二：实现 Rerank 精排重打分与强制引用溯源问答引擎

- **目标文件**：`versions/day60/src/day60_rag_pipeline.py`
- **老陈代码旁白**：粗排拿到候选切片后，加上 Rerank 精排、低分防幻觉拒答门禁和强制 `[DOC-xx]` 角标引用，组装出完整的星辰智脑 RAG 问答服务。


In [ ]:
"""
Day 60 - 模块二：Rerank 精排 + 置信度拒答 + 强制 [doc_id] 引用溯源问答流水线
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List

from src.day60_hybrid_retriever import HybridRetriever, KnowledgeChunk, cosine_sim, semantic_expand, tokenize


@dataclass
class Citation:
    doc_id: str
    title: str
    rerank_score: float
    snippet: str


@dataclass
class RAGAnswer:
    query: str
    answered: bool
    answer: str
    citations: List[Citation] = field(default_factory=list)


def rerank_score(query: str, chunk: KnowledgeChunk) -> float:
    """精排重打分 [0.0, 1.0]：融合语义余弦 + 标题命中加权 + 关键词覆盖率"""
    q_toks = tokenize(query)
    if not q_toks:
        return 0.0
    title_toks = set(tokenize(chunk.title))
    body_toks = set(tokenize(f"{chunk.title} {chunk.content}"))

    sem = cosine_sim(semantic_expand(q_toks), semantic_expand(list(body_toks)))
    title_hit = sum(1 for t in set(q_toks) if t in title_toks) / max(1, len(set(q_toks)))
    body_hit = sum(1 for t in set(q_toks) if t in body_toks) / max(1, len(set(q_toks)))
    code_tokens = [t for t in set(q_toks) if t.isascii() and any(ch.isdigit() for ch in t)]
    exact_code_boost = 0.35 if (code_tokens and any(t in body_toks for t in code_tokens)) else 0.0
    score = 0.50 * sem + 0.30 * title_hit + 0.20 * body_hit + exact_code_boost
    return round(min(1.0, score), 4)


class StarMindRAGEngine:
    """星辰智脑企业级 RAG 主链路引擎"""

    def __init__(self, chunks: List[KnowledgeChunk], min_confidence: float = 0.22) -> None:
        self.retriever = HybridRetriever(chunks)
        self.min_confidence = min_confidence

    def ask(self, query: str, top_k: int = 2) -> RAGAnswer:
        candidates = self.retriever.search_rrf(query, top_k=max(top_k * 2, 4))
        reranked = sorted(
            [(chunk, rerank_score(query, chunk)) for chunk, _ in candidates],
            key=lambda x: x[1],
            reverse=True,
        )
        if not reranked or reranked[0][1] < self.min_confidence:
            return RAGAnswer(
                query=query,
                answered=False,
                answer="抱歉，星辰知识库中未检索到足够可信的依据，为避免幻觉拒绝编造答案，请联系对应部门人工核实。",
                citations=[],
            )

        selected = [item for item in reranked[:top_k] if item[1] >= self.min_confidence * 0.8]
        citations = [
            Citation(c.doc_id, c.title, sc, c.content[:80])
            for c, sc in selected
        ]
        summary_lines = [f"- {c.content} [{c.doc_id}]" for c, _ in selected]
        answer_text = "根据星辰数科企业知识库检索结果，为您整理如下依据：\n" + "\n".join(summary_lines)
        return RAGAnswer(query=query, answered=True, answer=answer_text, citations=citations)


if __name__ == "__main__":
    kb = [
        KnowledgeChunk("DOC-01", "差旅住宿报销标准", "一线城市出差住宿上限650元/晚，需在费用中心上传电子专票核销。"),
        KnowledgeChunk("DOC-02", "网关错误码ERR_4291手册", "错误码ERR_4291表示部门日Token配额已触发100%硬熔断，请联系管理员提额。"),
    ]
    rag = StarMindRAGEngine(kb)
    print("命中场景:", rag.ask("出差住宾馆报销标准是什么？"))
    print("拒答场景:", rag.ask("火星移民飞船票价多少？"))


### 🔍 步骤 2 代码深度解析

把精排得分严格归一化到 `[0, 1]`，低于 `min_confidence` 时主动触发防幻觉拒答；命中时每条事实均自动拼接 `[DOC-xx]` 引用角标与原文切片元数据。

**预期运行输出参考：**
```text
命中场景: RAGAnswer(query='出差住宾馆报销标准是什么？', answered=True, answer='根据星辰数科企业知识库检索结果... [DOC-01]', citations=[Citation(...)])
拒答场景: RAGAnswer(query='火星移民飞船票价多少？', answered=False, answer='抱歉，星辰知识库中未检索到足够可信的依据...', citations=[])
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day60/tests/test_day60.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 60 单元测试：混合检索 RAG + Rerank + 引用溯源"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day60_hybrid_retriever import KnowledgeChunk
from src.day60_rag_pipeline import StarMindRAGEngine

SAMPLE_KB = [
    KnowledgeChunk("DOC-01", "差旅住宿报销标准", "一线城市出差住宿上限650元/晚，需在费用中心上传电子专票核销。"),
    KnowledgeChunk("DOC-02", "网关错误码ERR_4291手册", "错误码ERR_4291表示部门日Token配额已触发100%硬熔断，请联系管理员提额。"),
    KnowledgeChunk("DOC-03", "新员工飞书SSO登录指引", "新员工需管理员同步组织架构并分配基础角色后方可飞书扫码登录。"),
]


def test_rag_hits_exact_error_code_with_citation():
    rag = StarMindRAGEngine(SAMPLE_KB)
    res = rag.ask("调用接口报错 ERR_4291 怎么解决？")
    assert res.answered is True
    assert res.citations[0].doc_id == "DOC-02"
    assert "[DOC-02]" in res.answer


def test_rag_hits_synonym_query():
    rag = StarMindRAGEngine(SAMPLE_KB)
    res = rag.ask("出差住宾馆最多花多少钱？")
    assert res.answered is True
    assert res.citations[0].doc_id == "DOC-01"


def test_rag_refuses_out_of_domain_question():
    rag = StarMindRAGEngine(SAMPLE_KB)
    res = rag.ask("宇宙大爆炸奇点密度如何计算？")
    assert res.answered is False
    assert res.citations == []

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：为知识库增加带重叠窗口的滑动切片器

**题目要求：**
编写 `chunk_document(doc_id, title, text, chunk_size=120, overlap=20) -> list[KnowledgeChunk]`，将长文档切分为带序号子切片（如 `DOC-01#1`）。

**思路提示：** 步长 `step = chunk_size - overlap`，循环切片直到末尾。


In [ ]:
from src.day60_hybrid_retriever import KnowledgeChunk


def chunk_document(doc_id: str, title: str, text: str, chunk_size: int = 120, overlap: int = 20):
    step = max(1, chunk_size - overlap)
    chunks = []
    for idx, start in enumerate(range(0, len(text), step), start=1):
        piece = text[start : start + chunk_size].strip()
        if piece:
            chunks.append(KnowledgeChunk(f"{doc_id}#{idx}", title, piece))
    return chunks


res = chunk_document("DOC-99", "规章", "这是一段较长的企业制度文本。" * 15, chunk_size=50, overlap=10)
print(f"切分出 {len(res)} 个带重叠窗口切片，首片ID={res[0].doc_id}")


**答案解析与评分标准：**

适当重叠可防止关键句恰好落在两个切块边界被截断。


### 作业 2（企业实战）：企业实战：按部门权限标签（RBAC）过滤检索结果

**题目要求：**
实现前置权限过滤：只有当文档的 `allowed_depts` 包含用户所属部门或包含 `'ALL'` 时，才允许进入检索候选池，防止普通员工搜出高管薪酬制度。

**思路提示：** 在构造 `HybridRetriever` 前或在检索结果返回前按 `user_dept in doc.allowed_depts` 过滤。


In [ ]:
def filter_chunks_by_dept(docs: list[dict], user_dept: str) -> list[dict]:
    return [d for d in docs if "ALL" in d["allowed_depts"] or user_dept in d["allowed_depts"]]


docs = [
    {"id": "D1", "allowed_depts": {"ALL"}},
    {"id": "D2", "allowed_depts": {"HR_ADMIN"}},
]
print("普通研发可见文档:", [d["id"] for d in filter_chunks_by_dept(docs, "RD")])


**答案解析与评分标准：**

企业 RAG 的权限隔离必须做在检索层（Pre-filtering），绝不能让敏感切片进了 Prompt 再指望模型闭嘴。


### 作业 3（架构进阶）：架构进阶：计算 Recall@K 与 MRR 检索质量指标

**题目要求：**
编写 `eval_retrieval(retrieved_ids_list, expected_ids)` 计算检索集上的 `Recall@3` 与 `MRR`（平均倒数排名）。

**思路提示：** 若目标 doc_id 出现在第 `r` 位（1-indexed），该题 MRR 贡献 `1/r`，未命中贡献 0。


In [ ]:
def eval_retrieval(retrieved_list: list[list[str]], expected_list: list[str], k: int = 3) -> dict:
    hits, rr_sum = 0, 0.0
    for ret, exp in zip(retrieved_list, expected_list):
        top_k = ret[:k]
        if exp in top_k:
            hits += 1
            rr_sum += 1.0 / (top_k.index(exp) + 1)
    n = max(1, len(expected_list))
    return {"recall_at_k": round(hits / n, 4), "mrr": round(rr_sum / n, 4)}


print(eval_retrieval([["D1", "D2"], ["D3", "D1"], ["D2", "D3"]], ["D1", "D1", "D4"]))


**答案解析与评分标准：**

Recall@K 看找不找得到，MRR 看排不排得靠前，这两项是 Day 63 Ragas 评估的核心前置检索指标。


---

## 🎯 老陈收尾复盘

漂亮！星辰智脑的核心 RAG 引擎全部跑通单测：专有错误码靠 BM25 精准命中，同义语义靠向量召回，RRF 融合 + Rerank 精排把最相关条款顶到 Top-1，而且每条回答都带着可点击溯源的 `[DOC-xx]` 引用角标，彻底堵死胡编乱造！明天咱把它接入 LangGraph 多 Agent！
